In [5]:
import numpy as np
import scipy.io as scio
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [3]:
from importlib import reload
import my_BLS_Regression_5
reload(my_BLS_Regression_5)
from my_BLS_Regression_5 import BLS_regression

In [7]:
data, data_X, data_y = [],[],[]
file_path = "D:\\1 ارشد\پایان نامه\\گزارش 23 - ارائه پروپوزال\\TE Data\\"

for mode in ['1','2','3','4','5','6']:
    mode_name = 'm' + mode + 'd00'
    temp = scio.loadmat(file_path + mode_name)[mode_name]
    #data.append(temp.copy())
    data_X.append(temp[:,[0, 1, 2, 3, 4, 5, 8, 9, 10, 12, 13, 15, 17, 18, 20, 21]].copy())
    data_y.append(temp[:,32].copy())  #purge E

np.round(pd.DataFrame(data_X[1]),3).head()


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,0.307,737.367,8033.449,8.561,31.701,46.110,124.205,0.354,90.306,2705.738,26.315,3328.012,65.418,4.898,108.524,91.509
1,0.306,731.896,8083.074,8.655,31.630,46.230,124.201,0.354,90.252,2705.751,26.224,3328.070,65.433,5.247,108.513,91.354
2,0.309,725.945,8033.404,8.579,31.647,46.043,124.195,0.353,90.237,2704.870,26.214,3327.154,65.427,4.299,108.524,91.433
3,0.309,751.991,8011.162,8.496,31.965,46.419,124.191,0.353,90.220,2705.372,26.371,3327.004,65.424,4.100,108.541,91.600
4,0.305,723.574,8042.783,8.588,31.571,46.129,124.201,0.354,90.245,2705.674,26.466,3327.606,65.413,5.565,108.532,91.634


In [8]:
data_X_scaled = []
scaler_0 = StandardScaler()
scaler_0.fit(data_X[0][:,:])
for mode in range(6):
    data_X_scaled.append(scaler_0.transform(data_X[mode][:,:]))


In [13]:
data_ar = np.hstack([data_X_scaled[0][:1000,:], data_y[0][:1000].reshape(-1,1)])
data_ar[0]

array([ 7.32353396e-01, -2.81374785e-01,  3.86273620e-01, -2.54189953e-01,
       -1.39937507e-01, -1.55715539e-01, -1.08762920e-01, -1.31671356e+00,
       -4.25718175e-01, -9.65568955e-01, -6.05866426e-02, -1.01803390e+00,
       -3.28082593e-01, -5.93257868e-04,  6.51475191e-01, -6.70948080e-01,
        1.61374012e+01])

In [15]:
index_permute = np.random.permutation(1000)

In [16]:
train1 = data_ar[index_permute[:750], :]
test1 = data_ar[index_permute[750:1000], :]

In [19]:
data_splitted = [data_ar[index_permute[:200],:], data_ar[index_permute[200:400],:], data_ar[index_permute[400:600],:], data_ar[index_permute[600:800],:], data_ar[index_permute[800:1000],:]]

fold = []

train1 = np.vstack([data_splitted[1], data_splitted[2], data_splitted[3], data_splitted[4]])
test1 = data_splitted[0]
fold.append([train1, test1])

train2 = np.vstack([data_splitted[0], data_splitted[2], data_splitted[3], data_splitted[4]])
test2 = data_splitted[1]
fold.append([train2, test2])

train3 = np.vstack([data_splitted[0], data_splitted[1], data_splitted[3], data_splitted[4]])
test3 = data_splitted[2]
fold.append([train3, test3])

train4 = np.vstack([data_splitted[0], data_splitted[1], data_splitted[2], data_splitted[4]])
test4 = data_splitted[3]
fold.append([train4, test4])

train5 = np.vstack([data_splitted[1], data_splitted[2], data_splitted[3], data_splitted[0]])
test5 = data_splitted[4]
fold.append([train5, test5])


In [20]:
fold[1][1].shape, fold[1][0].shape

((200, 17), (800, 17))

In [28]:
fit_RMSE_sum, fit_MAPE_sum, pred_RMSE_sum, pred_MAPE_sum = [],[],[],[]

for traintest in fold:
    
    model = BLS_regression(0.001, 1, 10, 8, 6, 3, activation='relu6')
    model.fit(traintest[0][:,:16], traintest[0][:,16])
    model.predict(traintest[1][:,:16], traintest[1][:,16])
    fit_RMSE_sum.append(model.fit_RMSE)
    fit_MAPE_sum.append(model.fit_MAPE)
    pred_RMSE_sum.append(model.pred_RMSE)
    pred_MAPE_sum.append(model.pred_MAPE)

In [34]:
[i*100 for i in fit_MAPE_sum]

[1.1270500656015225,
 1.1283218968796827,
 1.1087342579307342,
 1.1534112256686675,
 1.0916479707837101]

In [36]:
fit_RMSE = np.array(fit_RMSE_sum)
pred_RMSE = np.array(pred_RMSE_sum)
fit_MAPE = np.array(fit_MAPE_sum)
pred_MAPE = np.array(pred_MAPE_sum)

In [38]:
np.mean(fit_RMSE), np.max(fit_RMSE) - np.mean(fit_RMSE), np.min(fit_RMSE) - np.mean(fit_RMSE)

(0.22451589261043123, 0.005121586080280294, -0.005163291850077606)

In [39]:
np.mean(pred_RMSE), np.max(pred_RMSE) - np.mean(pred_RMSE), np.min(pred_RMSE) - np.mean(pred_RMSE)

(0.2325653750419095, 0.0194999743545875, -0.021846742988813173)

In [42]:
np.mean(fit_MAPE)*100, (np.max(fit_MAPE) - np.mean(fit_MAPE)), (np.min(fit_MAPE) - np.mean(fit_MAPE))*100

(1.1218330833728636, 0.0003157814229580395, -0.030185112589153386)

In [ ]:
np.mean(pred_MAPE)*100, (np.max(pred_MAPE) - np.mean(pred_MAPE))*100, (np.min(pred_MAPE) - np.mean(pred_MAPE))*100

(0.011652608455290774, 0.0012592460373745534, -0.0013153820154647323)